In [ ]:
# ============================================================
# 2023.01 ~ 2025.12 동안 공백이 한 번도 없는 법인 추출
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. 엑셀 불러오기
#    엑셀 파일 이름을 im_bank_data.xlsx로 가정, 실행 코드 파일과 엑셀 파일이 같은 폴더 위치에 있다고 가정***
# ------------------------------------------------------------

FILE_PATH = "im_bank_data.xlsx"

df = pd.read_excel(
    FILE_PATH,
    sheet_name="Sheet1",
    engine="openpyxl"
)

print("===== 원본 데이터 =====")
print(f"행 수: {len(df):,}")
print(f"열 수: {len(df.columns):,}")
print(f"고유 법인 수: {df['법인ID'].nunique():,}")


# ------------------------------------------------------------
# 2. 기준년월 정리
# ------------------------------------------------------------

df["기준년월"] = pd.to_numeric(
    df["기준년월"],
    errors="raise"
).astype(int)

print("\n===== 기간 확인 =====")
print("최소 기준년월:", df["기준년월"].min())
print("최대 기준년월:", df["기준년월"].max())
print("전체 고유 월 수:", df["기준년월"].nunique())


# ------------------------------------------------------------
# 3. 분석기간 36개월 생성
#    2023.01 ~ 2025.12
# ------------------------------------------------------------

expected_months = (
    pd.period_range(
        start="2023-01",
        end="2025-12",
        freq="M"
    )
    .strftime("%Y%m")
    .astype(int)
)

print("\n===== 기대 기간 =====")
print("시작:", expected_months[0])
print("종료:", expected_months[-1])
print("개월 수:", len(expected_months))


# ------------------------------------------------------------
# 4. 법인별 관측 개월 수 계산
# ------------------------------------------------------------

corp_month_count = (
    df.groupby("법인ID")["기준년월"]
      .nunique()
)

print("\n===== 법인별 관측개월 수 =====")
print(f"최소: {corp_month_count.min()}")
print(f"중앙값: {corp_month_count.median():.1f}")
print(f"최대: {corp_month_count.max()}")


# ------------------------------------------------------------
# 5. 36개월 모두 관측된 법인 추출
# ------------------------------------------------------------

complete_corp_ids = corp_month_count[
    corp_month_count == 36
].index

# 해당 법인들의 행만 추출
complete_df = (
    df[df["법인ID"].isin(complete_corp_ids)]
    .copy()
    .sort_values(["법인ID", "기준년월"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. 최종 검증
# ------------------------------------------------------------

check_month_count = (
    complete_df.groupby("법인ID")["기준년월"]
               .nunique()
)

print("\n===== 공백이 한 번도 없는 법인 =====")
print(f"법인 수: {complete_df['법인ID'].nunique():,}")
print(f"행 수: {len(complete_df):,}")

print("\n===== 검증 =====")
print(
    "모든 법인이 정확히 36개월:",
    "PASS" if check_month_count.eq(36).all() else "CHECK"
)

print(
    "기대 법인 수 3,372:",
    "PASS" if complete_df["법인ID"].nunique() == 3372 else "CHECK"
)

print(
    "기대 행 수 121,392:",
    "PASS" if len(complete_df) == 3372 * 36 else "CHECK"
)


# ------------------------------------------------------------
# 7. 실제로 월 공백이 없는지도 최종 확인
# ------------------------------------------------------------

expected_month_set = set(expected_months)

all_months_complete = (
    complete_df.groupby("법인ID")["기준년월"]
    .apply(lambda x: set(x) == expected_month_set)
    .all()
)

print(
    "2023.01~2025.12 월이 하나도 빠지지 않음:",
    "PASS" if all_months_complete else "CHECK"
)

===== 원본 데이터 =====
행 수: 365,988
열 수: 70
고유 법인 수: 15,473

===== 기간 확인 =====
최소 기준년월: 202301
최대 기준년월: 202512
전체 고유 월 수: 36

===== 기대 기간 =====
시작: 202301
종료: 202512
개월 수: 36

===== 법인별 관측개월 수 =====
최소: 1
중앙값: 27.0
최대: 36

===== 공백이 한 번도 없는 법인 =====
법인 수: 3,372
행 수: 121,392

===== 검증 =====
모든 법인이 정확히 36개월: PASS
기대 법인 수 3,372: PASS
기대 행 수 121,392: PASS
2023.01~2025.12 월이 하나도 빠지지 않음: PASS


In [2]:
# ============================================================
# 36개월 완전관측은 아니지만,
# 가지고 있는 모든 년월이 연속된 법인 찾기
# ============================================================

# 기준년월을 월 단위 Period로 변환
temp = df[["법인ID", "기준년월"]].copy()

temp["month_period"] = pd.to_datetime(
    temp["기준년월"].astype(str),
    format="%Y%m"
).dt.to_period("M")


# ------------------------------------------------------------
# 법인별로 관측된 월이 연속인지 판정
# ------------------------------------------------------------

def check_consecutive_months(x):
    months = x.drop_duplicates().sort_values()

    # 월을 연속적인 정수값으로 변환
    month_numbers = months.astype("int64").to_numpy()

    # 1개월만 있어도 '가지고 있는 기간 내에서는 연속'으로 간주
    if len(month_numbers) <= 1:
        return True

    # 모든 인접 월의 차이가 정확히 1이면 연속
    return np.all(np.diff(month_numbers) == 1)


corp_consecutive = (
    temp.groupby("법인ID")["month_period"]
        .apply(check_consecutive_months)
)


# ------------------------------------------------------------
# 법인별 관측개월 수
# ------------------------------------------------------------

corp_month_count = (
    temp.groupby("법인ID")["month_period"]
        .nunique()
)


# ------------------------------------------------------------
# 36개월 완전관측은 제외하고,
# 나머지 중 내부 공백이 없는 법인
# ------------------------------------------------------------

partial_consecutive_mask = (
    (corp_month_count < 36)
    & (corp_consecutive)
)

partial_consecutive_ids = (
    partial_consecutive_mask[
        partial_consecutive_mask
    ].index
)


# 해당 법인 데이터
partial_consecutive_df = (
    df[df["법인ID"].isin(partial_consecutive_ids)]
    .copy()
    .sort_values(["법인ID", "기준년월"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 결과 출력
# ------------------------------------------------------------

print("===== 36개월 미만이지만 관측기간 내 공백 없는 법인 =====")
print(f"법인 수: {len(partial_consecutive_ids):,}")

print("\n===== 참고 =====")
print(f"전체 법인 수: {df['법인ID'].nunique():,}")
print(f"36개월 완전관측 법인: {(corp_month_count == 36).sum():,}")
print(f"36개월 미만 법인: {(corp_month_count < 36).sum():,}")

print(
    "36개월 미만 중 연속 관측 법인:",
    f"{partial_consecutive_mask.sum():,}"
)

print(
    "36개월 미만 중 중간 공백이 있는 법인:",
    f"{((corp_month_count < 36) & (~corp_consecutive)).sum():,}"
)


# ------------------------------------------------------------
# 연속 관측 법인들의 관측개월 수 분포
# ------------------------------------------------------------

print("\n===== 연속 관측 법인의 관측개월 수 분포 =====")

print(
    corp_month_count.loc[partial_consecutive_ids]
                    .value_counts()
                    .sort_index()
)

===== 36개월 미만이지만 관측기간 내 공백 없는 법인 =====
법인 수: 5,685

===== 참고 =====
전체 법인 수: 15,473
36개월 완전관측 법인: 3,372
36개월 미만 법인: 12,101
36개월 미만 중 연속 관측 법인: 5,685
36개월 미만 중 중간 공백이 있는 법인: 6,416

===== 연속 관측 법인의 관측개월 수 분포 =====
month_period
1     319
2     281
3     268
4     230
5     257
6     237
7     196
8     255
9     192
10    142
11    182
12    161
13    134
14    185
15    191
16    188
17    177
18    192
19    121
20    167
21    154
22     98
23    101
24     74
25     81
26    111
27    131
28    113
29    113
30    101
31     95
32    129
33     93
34     67
35    149
Name: count, dtype: int64


In [3]:
# ============================================================
# 36개월 완전관측 법인의 표본 대표성 점검
# ============================================================

import pandas as pd
import numpy as np

# scipy가 없다면:
# pip install scipy
from scipy.stats import chi2_contingency


# ============================================================
# 1. 전체 법인 중 36개월 완전관측 법인의 비율
# ============================================================

total_corp_n = df["법인ID"].nunique()
complete_corp_n = len(complete_corp_ids)
excluded_corp_n = total_corp_n - complete_corp_n

complete_ratio = complete_corp_n / total_corp_n * 100
excluded_ratio = excluded_corp_n / total_corp_n * 100


print("============================================================")
print("1. 36개월 완전관측 법인 비율")
print("============================================================")
print(f"전체 고유 법인 수       : {total_corp_n:,}")
print(f"36개월 완전관측 법인 수: {complete_corp_n:,}")
print(f"제외 법인 수            : {excluded_corp_n:,}")
print(f"완전관측 비율           : {complete_ratio:.2f}%")
print(f"제외 비율               : {excluded_ratio:.2f}%")

1. 36개월 완전관측 법인 비율
전체 고유 법인 수       : 15,473
36개월 완전관측 법인 수: 3,372
제외 법인 수            : 12,101
완전관측 비율           : 21.79%
제외 비율               : 78.21%


In [4]:
# ============================================================
# 2. 법인별 분석집단 표시
# ============================================================

corp_group = (
    df[["법인ID"]]
    .drop_duplicates()
    .copy()
)

corp_group["분석집단"] = np.where(
    corp_group["법인ID"].isin(complete_corp_ids),
    "36개월 완전관측",
    "제외군"
)

print("\n============================================================")
print("2. 분석집단별 법인 수")
print("============================================================")

print(corp_group["분석집단"].value_counts())


2. 분석집단별 법인 수
분석집단
제외군          12101
36개월 완전관측     3372
Name: count, dtype: int64


In [5]:
# ============================================================
# 3. 법인별 대표 특성 생성
#
# 관측기간 중 가장 자주 나타난 값(mode)을 해당 법인의 대표값으로 사용
# 최빈값이 여러 개면 가장 최근에 관측된 값을 선택
# ============================================================

compare_cols = [
    "업종_대분류",
    "업종_중분류",
    "사업장_시도",
    "사업장_시군구",
    "법인_고객등급",
    "전담고객여부"
]


# 실제 데이터에 존재하는 열만 사용
compare_cols = [col for col in compare_cols if col in df.columns]

print("비교 대상 변수:")
print(compare_cols)


def get_representative_value(group, col):
    """
    한 법인의 여러 월 자료 중
    해당 변수의 최빈값을 대표값으로 사용.
    최빈값이 여러 개면 가장 최근 관측값을 선택.
    """

    temp = group[["기준년월", col]].dropna()

    # 전부 결측이면
    if temp.empty:
        return "결측"

    # 값별 출현 횟수
    counts = temp[col].value_counts()

    # 최대 출현 횟수
    max_count = counts.max()

    # 최빈값 후보
    mode_candidates = counts[
        counts == max_count
    ].index.tolist()

    # 최빈값이 하나면 그대로
    if len(mode_candidates) == 1:
        return mode_candidates[0]

    # 동률이면 가장 최근에 등장한 값 선택
    temp = temp[
        temp[col].isin(mode_candidates)
    ].sort_values("기준년월")

    return temp.iloc[-1][col]


# 법인별 대표 특성 데이터프레임 생성
corp_characteristics = pd.DataFrame(
    {"법인ID": df["법인ID"].unique()}
)

for col in compare_cols:

    representative = (
        df.groupby("법인ID", group_keys=False)
          .apply(
              lambda g: get_representative_value(g, col),
              include_groups=False
          )
          .rename(col)
          .reset_index()
    )

    corp_characteristics = corp_characteristics.merge(
        representative,
        on="법인ID",
        how="left"
    )


# 완전관측/제외 집단 정보 결합
corp_characteristics = corp_characteristics.merge(
    corp_group,
    on="법인ID",
    how="left"
)


print("\n============================================================")
print("3. 법인 단위 대표 특성 데이터")
print("============================================================")
print(corp_characteristics.head())

print("\n법인 수 검증:")
print("전체 법인 수 :", corp_characteristics["법인ID"].nunique())
print("행 수        :", len(corp_characteristics))

In [6]:
# ============================================================
# 4. 완전관측군 vs 제외군 분포 비교
# ============================================================

distribution_results = {}


for col in compare_cols:

    print("\n\n")
    print("=" * 70)
    print(f"[{col}]")
    print("=" * 70)

    temp = corp_characteristics[
        ["법인ID", "분석집단", col]
    ].copy()

    temp[col] = temp[col].fillna("결측").astype(str)


    # --------------------------------------------------------
    # 빈도표
    # --------------------------------------------------------

    count_table = pd.crosstab(
        temp[col],
        temp["분석집단"]
    )


    # --------------------------------------------------------
    # 각 집단 내부 비율(%)
    # --------------------------------------------------------

    pct_table = (
        pd.crosstab(
            temp[col],
            temp["분석집단"],
            normalize="columns"
        ) * 100
    )


    # --------------------------------------------------------
    # 완전관측군 - 제외군 비율 차이 (%p)
    # --------------------------------------------------------

    if (
        "36개월 완전관측" in pct_table.columns
        and "제외군" in pct_table.columns
    ):

        pct_table["차이(%p)"] = (
            pct_table["36개월 완전관측"]
            - pct_table["제외군"]
        )


    # 보기 좋게 반올림
    pct_table = pct_table.round(2)


    print("\n[법인 수]")
    print(count_table)

    print("\n[집단 내 구성비 및 차이]")
    print(pct_table)


    # 저장
    distribution_results[col] = {
        "count": count_table,
        "percentage": pct_table
    }




[업종_대분류]

[법인 수]
분석집단                      36개월 완전관측   제외군
업종_대분류                                   
건설업                             583  1506
광업                                9    14
교육 서비스업                          31   181
금융 및 보험업                         23   141
농업, 임업 및 어업                       5    75
도매 및 소매업                        706  2213
보건업 및 사회복지 서비스업                  19    94
부동산업                            269  1091
사업시설 관리, 사업 지원 및 임대 서비스업        107   399
수도, 하수 및 폐기물 처리, 원료 재생업          54   127
숙박 및 음식점업                        60   393
예술, 스포츠 및 여가관련 서비스업              28   191
운수 및 창고업                        142   369
전기, 가스, 증기 및 공기조절 공급업            25    57
전문, 과학 및 기술 서비스업                 83   407
정보통신업                            53   359
제조업                            1093  4212
협회 및 단체, 수리 및 기타 개인 서비스업         82   272

[집단 내 구성비 및 차이]
분석집단                      36개월 완전관측    제외군  차이(%p)
업종_대분류                                            
건설업                  

In [7]:
# ============================================================
# 5. Cramér's V
#    완전관측 여부와 각 범주형 특성의 관련성 측정
# ============================================================

def cramers_v(confusion_matrix):

    chi2 = chi2_contingency(confusion_matrix)[0]

    n = confusion_matrix.to_numpy().sum()

    r, k = confusion_matrix.shape

    denominator = min(k - 1, r - 1)

    if denominator == 0:
        return np.nan

    return np.sqrt(
        chi2 / (n * denominator)
    )


effect_results = []


for col in compare_cols:

    temp = corp_characteristics[
        ["분석집단", col]
    ].copy()

    temp[col] = temp[col].fillna("결측").astype(str)

    contingency = pd.crosstab(
        temp["분석집단"],
        temp[col]
    )


    # 집단 또는 범주가 하나뿐이면 계산 불가능
    if min(contingency.shape) < 2:

        chi2 = np.nan
        p_value = np.nan
        cv = np.nan

    else:

        chi2, p_value, dof, expected = chi2_contingency(
            contingency
        )

        cv = cramers_v(contingency)


    effect_results.append({
        "변수": col,
        "Cramers_V": cv,
        "Chi_square": chi2,
        "p_value": p_value
    })


effect_df = pd.DataFrame(effect_results)

effect_df = effect_df.sort_values(
    "Cramers_V",
    ascending=False
).reset_index(drop=True)


print("\n============================================================")
print("5. 완전관측 여부와 특성 간 관련성")
print("============================================================")

print(
    effect_df.round({
        "Cramers_V": 4,
        "Chi_square": 2,
        "p_value": 6
    })
)


5. 완전관측 여부와 특성 간 관련성
        변수  Cramers_V  Chi_square  p_value
0  법인_고객등급     0.1750      473.72      0.0
1   전담고객여부     0.1578      385.35      0.0
2   업종_중분류     0.1425      314.13      0.0
3  사업장_시군구     0.1329      273.15      0.0
4   사업장_시도     0.1274      251.17      0.0
5   업종_대분류     0.1049      170.22      0.0


In [8]:
# ============================================================
# 6. 최종 요약
# ============================================================

print("\n" + "=" * 70)
print("표본 대표성 점검 요약")
print("=" * 70)

print(f"""
전체 고유 법인 수        : {total_corp_n:,}
36개월 완전관측 법인 수 : {complete_corp_n:,}
제외 법인 수             : {excluded_corp_n:,}

36개월 완전관측 비율     : {complete_ratio:.2f}%
제외 비율                : {excluded_ratio:.2f}%
""")

print("[완전관측 여부와 기본 특성의 관련성]")
print(
    effect_df[
        ["변수", "Cramers_V", "p_value"]
    ].round(4)
)


표본 대표성 점검 요약

전체 고유 법인 수        : 15,473
36개월 완전관측 법인 수 : 3,372
제외 법인 수             : 12,101

36개월 완전관측 비율     : 21.79%
제외 비율                : 78.21%

[완전관측 여부와 기본 특성의 관련성]
        변수  Cramers_V  p_value
0  법인_고객등급     0.1750      0.0
1   전담고객여부     0.1578      0.0
2   업종_중분류     0.1425      0.0
3  사업장_시군구     0.1329      0.0
4   사업장_시도     0.1274      0.0
5   업종_대분류     0.1049      0.0
